# Candle Prediction using Market Depth

In [199]:
from pathlib import Path
import pandas as pd
import ast
import numpy as np
import datetime
from datetime import timedelta
from utils import resample_fractional_minute, apply_trailing_logic

In [200]:
# ---- Input ------
date_ = "21APR2026"
file_name = "NIFTY2642124400CE.xlsx"

file_path = Path(fr"D:\Study\Programs\trading\assets\logs\{date_}\extracted_symbols\{file_name}")
df = pd.read_excel(file_path)
if "PE" in file_name or "CE" in file_name:
    col_name = "last_trade_time"

    # Volume computation
    # 1. Calculate the basic difference between rows
    df["volume_momentary"] = df["volume_traded"].diff()
    df.loc[df["volume_momentary"] == 0, "volume_momentary"] = np.nan
    df["volume_momentary"] = df["volume_momentary"].ffill()
    df["volume_momentary"] = df["volume_momentary"].fillna(0)
else:
    col_name = "local_time"

df[col_name] = pd.to_datetime(df[col_name])
target_date = pd.to_datetime(date_).date()
df = df[df[col_name].dt.date == target_date]

In [201]:
file_name

'NIFTY2642124400CE.xlsx'

In [202]:
df.head(4)

,instrument_token,symbol,exchange_timestamp,local_time,last_trade_time,last_price,last_traded_quantity,average_traded_price,option_CE_PE,option_type,...,total_sell_quantity,ohlc,change,oi,oi_day_high,oi_day_low,depth,tradable,mode,volume_momentary
1,16239618,NIFTY2642124400CE,NaN,2026-04-21 09:15:00.541,2026-04-21 09:15:00,74.35,130,84.38,CE,atm_plus_1.0,...,60580,"{'open': 100.0, 'high': 105.95, 'low': 68.1, '...",-29.825389,5268120,5268120,5268120,"{'buy': [{'quantity': 520, 'price': 72.95, 'or...",True,full,9230.0
2,16239618,NIFTY2642124400CE,NaN,2026-04-21 09:15:01.547,2026-04-21 09:15:00,55.25,260,84.38,CE,atm_plus_1.0,...,60580,"{'open': 100.0, 'high': 105.95, 'low': 68.1, '...",-47.852761,5268120,5268120,5268120,"{'buy': [{'quantity': 520, 'price': 72.95, 'or...",True,full,9230.0
3,16239618,NIFTY2642124400CE,NaN,2026-04-21 09:15:02.042,2026-04-21 09:15:01,58.65,65,60.24,CE,atm_plus_1.0,...,177905,"{'open': 100.0, 'high': 105.95, 'low': 53.25, ...",-44.643700,5268120,5268120,5268120,"{'buy': [{'quantity': 650, 'price': 58.25, 'or...",True,full,128310.0
4,16239618,NIFTY2642124400CE,NaN,2026-04-21 09:15:02.541,2026-04-21 09:15:01,61.40,195,60.24,CE,atm_plus_1.0,...,177905,"{'open': 100.0, 'high': 105.95, 'low': 53.25, ...",-42.048136,5268120,5268120,5268120,"{'buy': [{'quantity': 650, 'price': 58.25, 'or...",True,full,128310.0


In [203]:
# ---- Input ------
N = 6
window = 3
threshold = 0.005

In [204]:
df.columns

Index(['instrument_token', 'symbol', 'exchange_timestamp', 'local_time',
       'last_trade_time', 'last_price', 'last_traded_quantity',
       'average_traded_price', 'option_CE_PE', 'option_type', 'strike',
       'volume_traded', 'total_buy_quantity', 'total_sell_quantity', 'ohlc',
       'change', 'oi', 'oi_day_high', 'oi_day_low', 'depth', 'tradable',
       'mode', 'volume_momentary'],
      dtype='str')

In [205]:
print(type(df.iloc[0]["local_time"]))
print(df.iloc[0]["local_time"])
print(df.iloc[0]["last_trade_time"])

<class 'pandas.Timestamp'>
2026-04-21 09:15:00.541000
2026-04-21 09:15:00


In [206]:
clubbed_df = resample_fractional_minute(df, col_name, N)
clubbed_df["bucket_time_next"] = clubbed_df["bucket_time"].shift(-1)
clubbed_df["price_diff"] = clubbed_df["close"] - clubbed_df["open"]
clubbed_df["price_pct"] = (clubbed_df["close"] - clubbed_df["open"])/clubbed_df["open"]

clubbed_df["volume_diff"] = clubbed_df["volume_close"] - clubbed_df["volume_open"]
clubbed_df["volume_pct"] = (clubbed_df["volume_close"] - clubbed_df["volume_open"])/clubbed_df["volume_open"]


In [207]:
clubbed_df[clubbed_df['bucket_time'].dt.minute == 0].head()

,bucket_time,minute,open,high,low,close,volume_clubbed,volume_open,volume_high,volume_low,...,candle_type,minute_open,minute_high,minute_low,minute_close,bucket_time_next,price_diff,price_pct,volume_diff,volume_pct
270,2026-04-21 10:00:00,2026-04-21 10:00:00,128.85,130.00,128.50,129.60,400660.0,75140.0,112450.0,41600.0,...,BUY,128.85,131.75,125.35,129.9,2026-04-21 10:00:10,0.75,0.005821,-24960.0,-0.332180
271,2026-04-21 10:00:10,2026-04-21 10:00:00,129.60,129.80,127.70,128.10,181870.0,26650.0,36335.0,20670.0,...,BUY,128.85,131.75,125.35,129.9,2026-04-21 10:00:20,-1.50,-0.011574,-5980.0,-0.224390
272,2026-04-21 10:00:20,2026-04-21 10:00:00,130.00,131.75,129.25,129.65,329940.0,21060.0,117845.0,21060.0,...,BUY,128.85,131.75,125.35,129.9,2026-04-21 10:00:30,-0.35,-0.002692,5785.0,0.274691
273,2026-04-21 10:00:30,2026-04-21 10:00:00,129.85,130.55,125.35,125.35,239720.0,28860.0,66560.0,21450.0,...,BUY,128.85,131.75,125.35,129.9,2026-04-21 10:00:40,-4.50,-0.034655,37700.0,1.306306
274,2026-04-21 10:00:40,2026-04-21 10:00:00,125.50,128.25,125.50,128.25,122525.0,26650.0,34840.0,12285.0,...,BUY,128.85,131.75,125.35,129.9,2026-04-21 10:00:50,2.75,0.021912,-13130.0,-0.492683


In [208]:
clubbed_df[["volume_open", "volume_high", "volume_low", "volume_close", "volume_clubbed"]]

,volume_open,volume_high,volume_low,volume_close,volume_clubbed
0,9230.0,194545.0,9230.0,170170.0,970840.0
1,210795.0,403260.0,200135.0,321945.0,1590095.0
2,331370.0,331370.0,260975.0,266955.0,1726920.0
3,320905.0,320905.0,166920.0,166920.0,1481285.0
4,233870.0,284505.0,143715.0,143715.0,1292135.0
...,...,...,...,...,...
2155,1430.0,3445.0,195.0,1885.0,8580.0
2156,15340.0,15340.0,910.0,910.0,30680.0
2157,455.0,8255.0,455.0,1300.0,20280.0
2158,1950.0,3770.0,845.0,3770.0,12935.0


In [209]:
clubbed_df[clubbed_df['bucket_time'].dt.minute == 0].shape

(31, 21)

In [210]:
def generate_signal(
    df,
    window: int,
    threshold: float,
    volume_threshold: float = 0,

    use_price_level=True,
    use_price_trend=False,
    use_volume=False
):

    print("use_price_level : ", use_price_level)
    print("use_price_trend : ", use_price_trend)
    print("use_volume : ", use_volume)
    # ---------------------------
    # BASE CONDITIONS (level)
    # ---------------------------
    cond_buy = df["price_pct"] > threshold
    cond_sell = df["price_pct"] < -threshold

    if not use_price_level:
        cond_buy = pd.Series(True, index=df.index)
        cond_sell = pd.Series(True, index=df.index)

    # ---------------------------
    # PRICE TREND (monotonic)
    # ---------------------------
    if use_price_trend:
        price_diff = df["price_pct"].diff()

        cond_buy_trend = price_diff > 0
        cond_sell_trend = price_diff < 0

        buy_trend_streak = cond_buy_trend.rolling(window).sum() == window
        sell_trend_streak = cond_sell_trend.rolling(window).sum() == window
    else:
        buy_trend_streak = pd.Series(True, index=df.index)
        sell_trend_streak = pd.Series(True, index=df.index)

    # ---------------------------
    # VOLUME CONDITION
    # ---------------------------
    if use_volume:
        print("use_volume : ", use_volume)
        vol_diff = df["volume_pct"].diff()

        cond_vol = (df["volume_pct"] > volume_threshold) & (vol_diff > 0)
        vol_streak = cond_vol.rolling(window).sum() == window
    else:
        vol_streak = pd.Series(True, index=df.index)

    # ---------------------------
    # FINAL STREAKS
    # ---------------------------
    buy_streak = (
        cond_buy.rolling(window).sum() == window
    ) & buy_trend_streak & vol_streak

    sell_streak = (
        cond_sell.rolling(window).sum() == window
    ) & sell_trend_streak & vol_streak

    # ---------------------------
    # SIGNAL
    # ---------------------------
    df["predicted"] = np.where(
        buy_streak, "BUY",
        np.where(sell_streak, "SELL", None)
    )
    return df

In [211]:
clubbed_df_2 = generate_signal(clubbed_df, window, threshold, 0, use_price_level=True, use_price_trend=True, use_volume=False)

# PE/CE --- 0.005

use_price_level :  True
use_price_trend :  True
use_volume :  False


In [212]:
clubbed_df_2[clubbed_df_2["predicted"]==clubbed_df["candle_type"]].head(5)

,bucket_time,minute,open,high,low,close,volume_clubbed,volume_open,volume_high,volume_low,...,minute_open,minute_high,minute_low,minute_close,bucket_time_next,price_diff,price_pct,volume_diff,volume_pct,predicted
196,2026-04-21 09:47:40,2026-04-21 09:47:00,131.00,131.00,125.75,127.20,573300.0,71435.0,120770.0,70720.0,...,136.40,137.90,124.25,126.15,2026-04-21 09:47:50,-3.80,-0.029008,-715.0,-0.010009,SELL
304,2026-04-21 10:05:40,2026-04-21 10:05:00,136.25,137.05,133.55,133.55,250835.0,23205.0,80990.0,17160.0,...,139.65,141.45,131.55,136.75,2026-04-21 10:05:50,-2.70,-0.019817,13975.0,0.602241,SELL
380,2026-04-21 10:18:20,2026-04-21 10:18:00,131.15,131.15,127.70,128.50,237510.0,7410.0,70070.0,7410.0,...,132.80,133.60,127.70,128.20,2026-04-21 10:18:30,-2.65,-0.020206,36205.0,4.885965,SELL
393,2026-04-21 10:20:30,2026-04-21 10:20:00,121.45,121.50,119.60,119.70,222105.0,55055.0,95940.0,13325.0,...,124.95,124.95,118.55,124.15,2026-04-21 10:20:40,-1.75,-0.014409,40885.0,0.742621,SELL
475,2026-04-21 10:34:10,2026-04-21 10:34:00,132.25,135.45,132.25,135.00,197145.0,12740.0,70330.0,12740.0,...,130.55,145.60,130.35,144.00,2026-04-21 10:34:20,2.75,0.020794,28210.0,2.214286,BUY


In [213]:
# df_with_signal = df.merge(
#         clubbed_df[["bucket_time", "predicted"]],
#         left_on="last_trade_time",
#         right_on="bucket_time",
#         how="left"
#     )

import pandas as pd

# 1. Ensure both DataFrames are sorted by the time columns
df = df.sort_values("last_trade_time")
clubbed_df_2 = clubbed_df_2.sort_values("bucket_time")

# 2. Perform the proximity merge
df_with_signal = pd.merge_asof(
    df,
    clubbed_df_2[["bucket_time", "predicted"]],
    left_on="last_trade_time",
    right_on="bucket_time",
    direction="backward" # Only looks at the past/current, never the future
)

# Find duplicates in bucket_time and set their 'predicted' value to NaN
df_with_signal.loc[df_with_signal.duplicated(subset=['bucket_time'], keep='first'), 'predicted'] = np.nan

In [214]:
# df_with_signal.to_excel("df_with_signal.xlsx")

In [215]:
# clubbed_df_2.to_excel("clubbed_df_2.xlsx")

In [216]:
df_with_signal.head()

,instrument_token,symbol,exchange_timestamp,local_time,last_trade_time,last_price,last_traded_quantity,average_traded_price,option_CE_PE,option_type,...,change,oi,oi_day_high,oi_day_low,depth,tradable,mode,volume_momentary,bucket_time,predicted
0,16239618,NIFTY2642124400CE,NaN,2026-04-21 09:15:00.541,2026-04-21 09:15:00,74.35,130,84.38,CE,atm_plus_1.0,...,-29.825389,5268120,5268120,5268120,"{'buy': [{'quantity': 520, 'price': 72.95, 'or...",True,full,9230.0,2026-04-21 09:15:00,NaN
1,16239618,NIFTY2642124400CE,NaN,2026-04-21 09:15:01.547,2026-04-21 09:15:00,55.25,260,84.38,CE,atm_plus_1.0,...,-47.852761,5268120,5268120,5268120,"{'buy': [{'quantity': 520, 'price': 72.95, 'or...",True,full,9230.0,2026-04-21 09:15:00,NaN
2,16239618,NIFTY2642124400CE,NaN,2026-04-21 09:15:02.042,2026-04-21 09:15:01,58.65,65,60.24,CE,atm_plus_1.0,...,-44.643700,5268120,5268120,5268120,"{'buy': [{'quantity': 650, 'price': 58.25, 'or...",True,full,128310.0,2026-04-21 09:15:00,NaN
3,16239618,NIFTY2642124400CE,NaN,2026-04-21 09:15:02.541,2026-04-21 09:15:01,61.40,195,60.24,CE,atm_plus_1.0,...,-42.048136,5268120,5268120,5268120,"{'buy': [{'quantity': 650, 'price': 58.25, 'or...",True,full,128310.0,2026-04-21 09:15:00,NaN
4,16239618,NIFTY2642124400CE,NaN,2026-04-21 09:15:03.542,2026-04-21 09:15:03,61.80,260,61.11,CE,atm_plus_1.0,...,-41.670599,5268120,5268120,5268120,"{'buy': [{'quantity': 130, 'price': 61.7, 'ord...",True,full,194545.0,2026-04-21 09:15:00,NaN


In [217]:
df_with_signal.columns

Index(['instrument_token', 'symbol', 'exchange_timestamp', 'local_time',
       'last_trade_time', 'last_price', 'last_traded_quantity',
       'average_traded_price', 'option_CE_PE', 'option_type', 'strike',
       'volume_traded', 'total_buy_quantity', 'total_sell_quantity', 'ohlc',
       'change', 'oi', 'oi_day_high', 'oi_day_low', 'depth', 'tradable',
       'mode', 'volume_momentary', 'bucket_time', 'predicted'],
      dtype='str')

In [218]:
params = {
    "initial_sl_pct": 0.01,
    "initial_sl_price_pct": 0.02, #remove, in sweeping market we put an SL order.

    "target_pct": 0.02, # the threshold after which trailing behavior switches from loose → tight

    "trail_sl_pct": 0.02, # It is used only before till target_pct is hit. trail_sl_pct controls how much profit you allow to pull back while still staying in the trade.

    # Entry = 100
    # trail_sl_pct = 5% (0.05)
    #
    # Price = 102 → highest = 102
    # SL = 102 × 0.95 = 96.9
    #
    # Price = 108 → highest = 108
    # SL = 108 × 0.95 = 102.6

    "trail_price_pct": 0.10, # remove

    "tight_sl_offset": 2, # tight_sl_offset locks profits by keeping stop-loss a fixed distance below current price after target is reached.
    "tight_price_offset": 4, # remove

    "tight_sl_pct": 0.005 # remove
}

In [219]:
params = {
    "initial_sl_pct": 0.02, # 1% SL
    "target_pct": 0.01, # 2% profit threshold after which trailing behavior switches from loose → tight
    "trail_sl_pct": 0.02,
    "tight_sl_offset": 0.5, # tight_sl_offset locks profits by keeping stop-loss a fixed distance below current price after target is reached.
}

In [220]:
trades = apply_trailing_logic(df_with_signal, params)
trades = pd.DataFrame(trades)
if len(trades):
    trades["final"] = trades.apply(lambda row: "profit" if row["profit"] > 0 else "loss", axis=1)
else:
    print("trades not generated")


In [221]:
len(trades)

8

In [222]:
# trades

In [223]:
trades[trades["final"]=="profit"]["profit"].sum()

np.float64(17.399999999999977)

In [224]:
trades[trades["final"]=="loss"]["profit"].sum()

np.float64(0.0)

In [225]:
trades.head(11)

,entry_time,entry_price,exit_time,exit_price,profit,profit_pct,final
0,2026-04-21 10:34:11,132.25,2026-04-21 10:34:12,133.30,1.05,0.007940,profit
1,2026-04-21 10:34:21,135.50,2026-04-21 10:34:26,140.50,5.00,0.036900,profit
2,2026-04-21 12:00:30,143.75,2026-04-21 12:00:35,146.80,3.05,0.021217,profit
3,2026-04-21 12:10:11,157.50,2026-04-21 12:10:14,158.95,1.45,0.009206,profit
4,2026-04-21 12:55:50,146.80,2026-04-21 12:55:54,148.60,1.80,0.012262,profit
5,2026-04-21 13:15:41,149.80,2026-04-21 13:15:51,151.50,1.70,0.011348,profit
6,2026-04-21 13:50:11,138.30,2026-04-21 13:50:14,139.35,1.05,0.007592,profit
7,2026-04-21 14:12:51,135.00,2026-04-21 14:13:01,137.30,2.30,0.017037,profit


In [226]:
trades["final"].value_counts()

final
profit    8
Name: count, dtype: int64

In [227]:
trades[trades["final"]=="loss"].head(12)

,entry_time,entry_price,exit_time,exit_price,profit,profit_pct,final


In [228]:
import os
# clubbed_df_2.to_excel(Path(f"assets/logs/{date_}/clubbed_df.xlsx"))

In [229]:
clubbed_df_2["predicted"].value_counts()

predicted
SELL    12
BUY      8
Name: count, dtype: int64

In [230]:
from sklearn.metrics import confusion_matrix, classification_report

# Optional: remove rows without prediction
eval_df = clubbed_df_2.dropna(subset=["predicted", "candle_type"]).copy()

# If you want to ignore NO_SIGNAL:
eval_df = eval_df[eval_df["predicted"] != "NO_SIGNAL"]

y_true = eval_df["candle_type"]
y_pred = eval_df["predicted"]

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred, labels=["BUY", "SELL"])

print("Confusion Matrix (rows=true, cols=pred):")
print(cm)

# Detailed report
print("\nClassification Report:")
print(classification_report(y_true, y_pred))

# Precision : When the model says "BUY," how often is it actually a buy?
# Recall: Of all the actual "BUY" opportunities that happened, how many did the model catch?


Confusion Matrix (rows=true, cols=pred):
[[ 6  1]
 [ 2 11]]

Classification Report:
              precision    recall  f1-score   support

         BUY       0.75      0.86      0.80         7
        SELL       0.92      0.85      0.88        13

    accuracy                           0.85        20
   macro avg       0.83      0.85      0.84        20
weighted avg       0.86      0.85      0.85        20



In [231]:
# Filter BUY predictions
buy_preds = eval_df[eval_df["predicted"] == "BUY"]

# Counts
correct_buy = (buy_preds["candle_type"] == "BUY").sum()
incorrect_buy = (buy_preds["candle_type"] == "SELL").sum()

total_buy_preds = len(buy_preds)

# Precision (BUY accuracy relative to BUY predictions)
buy_precision = correct_buy / total_buy_preds if total_buy_preds > 0 else 0

print("Total BUY Predictions:", total_buy_preds)
print("Correct BUY Predictions:", correct_buy)
print("Incorrect BUY Predictions:", incorrect_buy)
print("BUY Precision:", buy_precision)

Total BUY Predictions: 8
Correct BUY Predictions: 6
Incorrect BUY Predictions: 2
BUY Precision: 0.75


In [232]:
# Filter SELL predictions
sell_preds = eval_df[eval_df["predicted"] == "SELL"]

# Counts
correct_sell = (sell_preds["candle_type"] == "SELL").sum()
incorrect_sell = (sell_preds["candle_type"] == "BUY").sum()

total_sell_preds = len(sell_preds)

# Precision (BUY accuracy relative to BUY predictions)
sell_precision = correct_sell / total_sell_preds if total_sell_preds > 0 else 0

print("Total SELL Predictions:", total_sell_preds)
print("Correct SELL Predictions:", correct_sell)
print("Incorrect SELL Predictions:", incorrect_sell)
print("SELL Precision:", sell_precision)

Total SELL Predictions: 12
Correct SELL Predictions: 11
Incorrect SELL Predictions: 1
SELL Precision: 0.9166666666666666
